# Custom JSON Decoding — Advanced Problems with Solutions
## Tutorial-Style Practice Notebook

This notebook is a second, independent set of advanced exercises on custom JSON decoding.

The style is deliberately tutorial-oriented:

- each problem starts from a concrete JSON document,
- we inspect what the standard decoder gives us,
- we identify the exact limitation,
- we solve one small part at a time,
- we test intermediate assumptions,
- and only then do we refactor into a cleaner design.

The main tools are:

- `json.loads`
- `json.load`
- `object_hook`
- `object_pairs_hook`
- `parse_float`
- `parse_int`
- `parse_constant`
- custom schemas / tagged objects
- `json.JSONDecoder`

All examples use only the Python standard library.


### Imports

We will use a few standard-library types so that our JSON data can be interpreted as richer Python objects.


In [1]:
import json

from collections import OrderedDict
from dataclasses import dataclass
from datetime import date, datetime, timedelta, timezone
from decimal import Decimal
from enum import Enum
from fractions import Fraction
from pathlib import Path
from types import MappingProxyType
from uuid import UUID


### A tiny helper for readable tests

The notebook uses simple assertions rather than a third-party testing framework.

This keeps every example copy-friendly and self-contained.


In [2]:
def check(condition, message="check failed"):
    if not condition:
        raise AssertionError(message)

print("Ready.")


Ready.


---

# Problem 1 — Why a Date-Looking String Is Still Just a String

Suppose an API sends us a report with a date:

```json
{
    "title": "Quarterly review",
    "published": "2026-07-31"
}
```

At first glance, the value of `published` *looks* like a date.

But JSON has no date type.

Let's begin by decoding it normally.


In [3]:
j = '''
{
    "title": "Quarterly review",
    "published": "2026-07-31"
}
'''

d = json.loads(j)
d


{'title': 'Quarterly review', 'published': '2026-07-31'}

If we inspect the type, we can see exactly what happened.


In [4]:
type(d["published"]), d["published"]


(str, '2026-07-31')

The decoder did nothing wrong.

The JSON token is a string, so Python correctly gave us a `str`.

If our application wants a `date`, then **our application needs a rule** that says when a string should be interpreted as a date.

A dangerous rule would be:

> "Whenever a string looks like YYYY-MM-DD, turn it into a date."

Why dangerous?

Because an ordinary product code, version identifier, or user-entered string could accidentally match that pattern.

A better approach is to make the JSON schema explicit.


### Step 1 — Add a type marker

We will encode a custom date as an object:

```json
{
    "__type__": "date",
    "value": "2026-07-31"
}
```

Now the JSON explicitly tells the decoder how the object should be interpreted.


In [5]:
j = '''
{
    "title": "Quarterly review",
    "published": {
        "__type__": "date",
        "value": "2026-07-31"
    }
}
'''

json.loads(j)


{'title': 'Quarterly review',
 'published': {'__type__': 'date', 'value': '2026-07-31'}}

The standard decoder still returns a nested dictionary.

Now we can add an `object_hook`.


In [6]:
def decode_date(obj):
    if obj.get("__type__") == "date":
        return date.fromisoformat(obj["value"])
    return obj


In [7]:
d = json.loads(j, object_hook=decode_date)
d


{'title': 'Quarterly review', 'published': datetime.date(2026, 7, 31)}

In [8]:
check(isinstance(d["published"], date))
check(d["published"] == date(2026, 7, 31))
print("Problem 1 solved.")


Problem 1 solved.


### Takeaway

Custom JSON decoding is not about discovering hidden Python types.

It is about defining an **interpretation protocol** for otherwise ordinary JSON values.


---

# Problem 2 — Observe the Bottom-Up Nature of `object_hook`

One of the most important behaviors of `object_hook` is that nested objects are processed before their parents.

Instead of just stating that fact, let's prove it.

Consider:


In [9]:
j = '''
{
    "level_1": {
        "level_2": {
            "level_3": {
                "value": 100
            }
        }
    }
}
'''


We will create a hook that only prints what it receives.


In [10]:
def trace_hook(obj):
    print("HOOK RECEIVED:", obj)
    return obj


In [11]:
d = json.loads(j, object_hook=trace_hook)


HOOK RECEIVED: {'value': 100}
HOOK RECEIVED: {'level_3': {'value': 100}}
HOOK RECEIVED: {'level_2': {'level_3': {'value': 100}}}
HOOK RECEIVED: {'level_1': {'level_2': {'level_3': {'value': 100}}}}


Notice the order:

1. the deepest dictionary is processed first,
2. then its parent,
3. then the next parent,
4. and finally the root object.

This matters because the parent can receive children that have **already been transformed**.


### Step 2 — Turn the deepest object into a custom value

Let's use a tagged fraction nested inside an ordinary object.


In [12]:
j = '''
{
    "experiment": {
        "result": {
            "__type__": "fraction",
            "numerator": 7,
            "denominator": 12
        }
    }
}
'''


In [13]:
def decode_fraction(obj):
    if obj.get("__type__") == "fraction":
        return Fraction(obj["numerator"], obj["denominator"])
    return obj

d = json.loads(j, object_hook=decode_fraction)
d


{'experiment': {'result': Fraction(7, 12)}}

By the time the `experiment` dictionary is processed, its `result` value is no longer a dictionary.

It is already a `Fraction`.


In [14]:
check(isinstance(d["experiment"]["result"], Fraction))
check(d["experiment"]["result"] == Fraction(7, 12))
print("Problem 2 solved.")


Problem 2 solved.


---

# Problem 3 — Build a Parent Object from Already-Decoded Children

Now let's use bottom-up decoding for something more useful.

We want to represent a time interval.

Our JSON schema will be:

```json
{
    "__type__": "interval",
    "start": {
        "__type__": "datetime",
        "value": "..."
    },
    "end": {
        "__type__": "datetime",
        "value": "..."
    }
}
```

The interesting part is that the `interval` decoder should receive actual `datetime` objects for `start` and `end`.


In [15]:
@dataclass(frozen=True)
class TimeInterval:
    start: datetime
    end: datetime

    @property
    def duration(self):
        return self.end - self.start


### Step 1 — Decode a single datetime


In [16]:
def decode_datetime(obj):
    if obj.get("__type__") == "datetime":
        value = obj["value"]
        if value.endswith("Z"):
            value = value[:-1] + "+00:00"
        return datetime.fromisoformat(value)
    return obj


### Step 2 — Extend the hook to decode an interval

The same hook is called for all objects.

Because children are processed first, we can check that `start` and `end` are already datetimes.


In [17]:
def decode_interval(obj):
    tag = obj.get("__type__")

    if tag == "datetime":
        value = obj["value"]
        if value.endswith("Z"):
            value = value[:-1] + "+00:00"
        return datetime.fromisoformat(value)

    if tag == "interval":
        start = obj["start"]
        end = obj["end"]

        if not isinstance(start, datetime):
            raise TypeError("interval.start was not decoded to datetime")

        if not isinstance(end, datetime):
            raise TypeError("interval.end was not decoded to datetime")

        if end < start:
            raise ValueError("interval end cannot be before start")

        return TimeInterval(start, end)

    return obj


In [18]:
j = '''
{
    "maintenance": {
        "__type__": "interval",
        "start": {
            "__type__": "datetime",
            "value": "2026-08-10T01:00:00Z"
        },
        "end": {
            "__type__": "datetime",
            "value": "2026-08-10T03:30:00Z"
        }
    }
}
'''

d = json.loads(j, object_hook=decode_interval)
d


{'maintenance': TimeInterval(start=datetime.datetime(2026, 8, 10, 1, 0, tzinfo=datetime.timezone.utc), end=datetime.datetime(2026, 8, 10, 3, 30, tzinfo=datetime.timezone.utc))}

In [19]:
check(isinstance(d["maintenance"], TimeInterval))
check(d["maintenance"].duration == timedelta(hours=2, minutes=30))
print("Duration:", d["maintenance"].duration)
print("Problem 3 solved.")


Duration: 2:30:00
Problem 3 solved.


### Takeaway

`object_hook` is especially convenient when parent custom objects contain other custom objects.

The parent decoder does not need to manually recurse into its children.


---

# Problem 4 — Use `parse_float` for Exact Financial Values

Suppose we receive:

```json
{
    "subtotal": 19.99,
    "tax": 1.60
}
```

By default, JSON decimal numbers become Python floats.


In [20]:
j = '''
{
    "subtotal": 19.99,
    "tax": 1.60
}
'''

d = json.loads(j)
d, type(d["subtotal"])


({'subtotal': 19.99, 'tax': 1.6}, float)

For many scientific calculations, binary floating-point is exactly what we want.

For decimal-domain values such as money, we may prefer `Decimal`.

The `parse_float` hook lets us decide how JSON floating-point tokens are constructed.


In [21]:
d = json.loads(j, parse_float=Decimal)
d


{'subtotal': Decimal('19.99'), 'tax': Decimal('1.60')}

In [22]:
type(d["subtotal"]), d["subtotal"] + d["tax"]


(decimal.Decimal, Decimal('21.59'))

### Step 2 — Verify that integers remain integers


In [23]:
j = '''
{
    "subtotal": 19.99,
    "tax": 1.60,
    "quantity": 3
}
'''

d = json.loads(j, parse_float=Decimal)

check(isinstance(d["subtotal"], Decimal))
check(isinstance(d["tax"], Decimal))
check(type(d["quantity"]) is int)

d


{'subtotal': Decimal('19.99'), 'tax': Decimal('1.60'), 'quantity': 3}

Only JSON tokens containing a decimal point or exponent use `parse_float`.

Integer tokens use the normal integer parser unless we also supply `parse_int`.


---

# Problem 5 — Learn What `parse_float` Actually Receives

Let's write a diagnostic function.

This is useful because it reveals a subtle detail: the hook receives the original numeric text, not a Python float.


In [24]:
def inspect_float(token):
    print("received:", repr(token), "type:", type(token))
    return Decimal(token)


In [25]:
json.loads('{"a": 0.125, "b": 1.2e3}', parse_float=inspect_float)


received: '0.125' type: <class 'str'>
received: '1.2e3' type: <class 'str'>


{'a': Decimal('0.125'), 'b': Decimal('1.2E+3')}

That design is important.

If the JSON parser converted the token to a float first, exact decimal information might already be lost.

Instead, our function receives text such as `'0.125'` or `'1.2e3'`.


---

# Problem 6 — Enforce a Numeric Range with `parse_int`

Imagine an external protocol that says every integer must fit in an unsigned 16-bit field.

Valid values are:

```text
0 through 65535
```

We can enforce that rule while the JSON integer token is being parsed.


In [26]:
def uint16(token):
    value = int(token)

    if not 0 <= value <= 65535:
        raise ValueError(f"{value} does not fit in uint16")

    return value


In [27]:
good = json.loads(
    '{"port": 443, "max_connections": 65535}',
    parse_int=uint16,
)

good


{'port': 443, 'max_connections': 65535}

Now try an invalid value.


In [28]:
bad = '{"port": 70000}'

try:
    json.loads(bad, parse_int=uint16)
except ValueError as ex:
    print("Rejected:", ex)


Rejected: 70000 does not fit in uint16


The advantage is that the policy applies to **every** JSON integer, including integers nested in arrays and nested objects.


---

# Problem 7 — Reject Python's Non-Standard JSON Constants

Python's standard decoder accepts:

- `NaN`
- `Infinity`
- `-Infinity`

These are useful extensions, but they are not part of strict JSON.

Let's see the default behavior.


In [29]:
json.loads('{"x": NaN, "y": Infinity, "z": -Infinity}')


{'x': nan, 'y': inf, 'z': -inf}

If we are building a strict API boundary, we may want to reject them.

That is what `parse_constant` is for.


In [30]:
def reject_constant(token):
    raise ValueError(f"non-standard JSON constant: {token}")


In [31]:
for text in [
    '{"x": NaN}',
    '{"x": Infinity}',
    '{"x": -Infinity}',
]:
    try:
        json.loads(text, parse_constant=reject_constant)
    except ValueError as ex:
        print("Rejected:", ex)


Rejected: non-standard JSON constant: NaN
Rejected: non-standard JSON constant: Infinity
Rejected: non-standard JSON constant: -Infinity


A normal JSON `null` is not sent to `parse_constant`.

It still becomes Python `None`.


In [32]:
d = json.loads(
    '{"x": null}',
    parse_constant=reject_constant,
)

check(d["x"] is None)
d


{'x': None}

---

# Problem 8 — Detect Duplicate Keys with `object_pairs_hook`

Consider:

```json
{
    "permission": "read",
    "permission": "admin"
}
```

What should that mean?

Different systems may treat duplicate keys differently.

A normal Python dictionary cannot represent both entries simultaneously.


In [33]:
j = '''
{
    "permission": "read",
    "permission": "admin"
}
'''

json.loads(j)


{'permission': 'admin'}

The later value replaced the earlier one.

If duplicate keys are invalid in our protocol, we need to see the key/value pairs *before* they become a dictionary.

That is exactly what `object_pairs_hook` provides.


In [34]:
def show_pairs(pairs):
    print(pairs)
    return dict(pairs)

json.loads(j, object_pairs_hook=show_pairs)


[('permission', 'read'), ('permission', 'admin')]


{'permission': 'admin'}

### Step 2 — Reject duplicates


In [35]:
def no_duplicate_keys(pairs):
    result = {}

    for key, value in pairs:
        if key in result:
            raise ValueError(f"duplicate key: {key!r}")
        result[key] = value

    return result


In [36]:
try:
    json.loads(j, object_pairs_hook=no_duplicate_keys)
except ValueError as ex:
    print("Rejected:", ex)


Rejected: duplicate key: 'permission'


---

# Problem 9 — Preserve an Object as a Read-Only Mapping

Sometimes we do not want a custom class at all.

Instead, we want every JSON object to become an immutable mapping.

`MappingProxyType` creates a read-only view of a dictionary.


### Step 1 — Try it with one object


In [37]:
def immutable_object(pairs):
    return MappingProxyType(dict(pairs))

d = json.loads(
    '{"a": 1, "b": 2}',
    object_pairs_hook=immutable_object,
)

d


mappingproxy({'a': 1, 'b': 2})

In [38]:
type(d)


mappingproxy

Attempting mutation should fail.


In [39]:
try:
    d["a"] = 100
except TypeError as ex:
    print("Mutation blocked:", ex)


Mutation blocked: 'mappingproxy' object does not support item assignment


### Step 2 — Notice what happens with nesting

Because the hook is applied bottom-up, nested objects are also converted.


In [40]:
d = json.loads(
    '{"outer": {"inner": {"x": 1}}}',
    object_pairs_hook=immutable_object,
)

type(d), type(d["outer"]), type(d["outer"]["inner"])


(mappingproxy, mappingproxy, mappingproxy)

---

# Problem 10 — Decode an Enum Safely

Suppose our application has three job states.


In [41]:
class JobStatus(Enum):
    PENDING = "pending"
    RUNNING = "running"
    COMPLETE = "complete"


We will use this JSON schema:

```json
{
    "__type__": "job_status",
    "value": "running"
}
```

The decoder should reject unknown values rather than silently keeping them.


In [42]:
def enum_decoder(obj):
    if obj.get("__type__") == "job_status":
        try:
            return JobStatus(obj["value"])
        except ValueError as ex:
            raise ValueError(
                f"unknown JobStatus value: {obj['value']!r}"
            ) from ex

    return obj


In [43]:
d = json.loads(
    '{"status": {"__type__": "job_status", "value": "running"}}',
    object_hook=enum_decoder,
)

d


{'status': <JobStatus.RUNNING: 'running'>}

In [44]:
check(d["status"] is JobStatus.RUNNING)
print("Problem 10 solved.")


Problem 10 solved.


Now test an invalid enum member.


In [45]:
try:
    json.loads(
        '{"status": {"__type__": "job_status", "value": "paused"}}',
        object_hook=enum_decoder,
    )
except ValueError as ex:
    print("Rejected:", ex)


Rejected: unknown JobStatus value: 'paused'


---

# Problem 11 — Decode `Path` Objects Without Treating All Strings as Paths

Paths are another good example of why explicit schemas are safer than guessing.

A string like:

```text
reports/2026/data.json
```

might be a filesystem path.

Or it might just be text.

So we will tag paths explicitly.


In [46]:
j = '''
{
    "output_directory": {
        "__type__": "path",
        "value": "reports/2026"
    },
    "description": "reports/2026"
}
'''


In [47]:
def path_decoder(obj):
    if obj.get("__type__") == "path":
        return Path(obj["value"])
    return obj

d = json.loads(j, object_hook=path_decoder)
d


{'output_directory': WindowsPath('reports/2026'),
 'description': 'reports/2026'}

In [48]:
check(isinstance(d["output_directory"], Path))
check(isinstance(d["description"], str))
print("Problem 11 solved.")


Problem 11 solved.


---

# Problem 12 — Decode a Dataclass with Validation

Let's model a geographic coordinate.

A valid coordinate must satisfy:

- latitude: -90 through 90
- longitude: -180 through 180

We will decode:

```json
{
    "__type__": "coordinate",
    "latitude": 42.6977,
    "longitude": 23.3219
}
```


In [49]:
@dataclass(frozen=True)
class Coordinate:
    latitude: Decimal
    longitude: Decimal


### Step 1 — Write the decoder

Because we want exact decimal tokens, we will combine `object_hook` with `parse_float=Decimal`.


In [50]:
def coordinate_decoder(obj):
    if obj.get("__type__") != "coordinate":
        return obj

    latitude = obj["latitude"]
    longitude = obj["longitude"]

    if not isinstance(latitude, Decimal):
        raise TypeError("latitude must be Decimal")

    if not isinstance(longitude, Decimal):
        raise TypeError("longitude must be Decimal")

    if not Decimal("-90") <= latitude <= Decimal("90"):
        raise ValueError("latitude is outside [-90, 90]")

    if not Decimal("-180") <= longitude <= Decimal("180"):
        raise ValueError("longitude is outside [-180, 180]")

    return Coordinate(latitude, longitude)


In [51]:
j = '''
{
    "city": "Sofia",
    "location": {
        "__type__": "coordinate",
        "latitude": 42.6977,
        "longitude": 23.3219
    }
}
'''

d = json.loads(
    j,
    object_hook=coordinate_decoder,
    parse_float=Decimal,
)

d


{'city': 'Sofia',
 'location': Coordinate(latitude=Decimal('42.6977'), longitude=Decimal('23.3219'))}

In [52]:
check(isinstance(d["location"], Coordinate))
check(d["location"].latitude == Decimal("42.6977"))
print("Problem 12 solved.")


Problem 12 solved.


### Step 2 — Test validation


In [53]:
bad = '''
{
    "__type__": "coordinate",
    "latitude": 120.0,
    "longitude": 23.0
}
'''

try:
    json.loads(
        bad,
        object_hook=coordinate_decoder,
        parse_float=Decimal,
    )
except ValueError as ex:
    print("Rejected:", ex)


Rejected: latitude is outside [-90, 90]


---

# Problem 13 — Refactor Multiple Custom Types into a Registry

So far, each decoder has been small.

But imagine a protocol supporting:

- date
- datetime
- fraction
- path
- UUID
- job status
- coordinate

A large `if/elif` chain would work, but it becomes harder to maintain.

A common improvement is a **decoder registry**.


### Step 1 — Make each type decoder responsible for one schema


In [54]:
def decode_date_record(obj):
    return date.fromisoformat(obj["value"])

def decode_datetime_record(obj):
    value = obj["value"]
    if value.endswith("Z"):
        value = value[:-1] + "+00:00"
    return datetime.fromisoformat(value)

def decode_fraction_record(obj):
    return Fraction(obj["numerator"], obj["denominator"])

def decode_path_record(obj):
    return Path(obj["value"])

def decode_uuid_record(obj):
    return UUID(obj["value"])

def decode_status_record(obj):
    return JobStatus(obj["value"])

def decode_coordinate_record(obj):
    latitude = obj["latitude"]
    longitude = obj["longitude"]

    if not Decimal("-90") <= latitude <= Decimal("90"):
        raise ValueError("invalid latitude")

    if not Decimal("-180") <= longitude <= Decimal("180"):
        raise ValueError("invalid longitude")

    return Coordinate(latitude, longitude)


### Step 2 — Build the registry


In [55]:
DECODERS = {
    "date": decode_date_record,
    "datetime": decode_datetime_record,
    "fraction": decode_fraction_record,
    "path": decode_path_record,
    "uuid": decode_uuid_record,
    "job_status": decode_status_record,
    "coordinate": decode_coordinate_record,
}


### Step 3 — Write one dispatcher


In [56]:
def registry_hook(obj):
    tag = obj.get("__type__")

    if tag is None:
        return obj

    decoder = DECODERS.get(tag)

    if decoder is None:
        return obj

    return decoder(obj)


Now the dispatcher does not need to know the details of each type.

It only chooses a decoder.


In [57]:
j = '''
{
    "id": {
        "__type__": "uuid",
        "value": "12345678-1234-5678-1234-567812345678"
    },
    "created": {
        "__type__": "datetime",
        "value": "2026-08-07T12:00:00Z"
    },
    "ratio": {
        "__type__": "fraction",
        "numerator": 3,
        "denominator": 8
    },
    "location": {
        "__type__": "coordinate",
        "latitude": 42.6977,
        "longitude": 23.3219
    }
}
'''

d = json.loads(
    j,
    object_hook=registry_hook,
    parse_float=Decimal,
)

d


{'id': UUID('12345678-1234-5678-1234-567812345678'),
 'created': datetime.datetime(2026, 8, 7, 12, 0, tzinfo=datetime.timezone.utc),
 'ratio': Fraction(3, 8),
 'location': Coordinate(latitude=Decimal('42.6977'), longitude=Decimal('23.3219'))}

---

# Problem 14 — Decide What to Do with Unknown Tagged Types

Our registry currently leaves unknown tagged dictionaries unchanged.

That is sometimes a useful forward-compatibility policy.

But in a strict internal protocol, an unknown type could mean:

- the producer is newer than the consumer,
- the payload is corrupted,
- or the wrong decoder was used.

Let's make the policy explicit.


In [58]:
def make_registry_hook(registry, *, unknown="keep"):
    if unknown not in {"keep", "error"}:
        raise ValueError("unknown must be 'keep' or 'error'")

    def hook(obj):
        tag = obj.get("__type__")

        if tag is None:
            return obj

        decoder = registry.get(tag)

        if decoder is not None:
            return decoder(obj)

        if unknown == "keep":
            return obj

        raise ValueError(f"unknown tagged type: {tag!r}")

    return hook


### Lenient mode


In [59]:
lenient_hook = make_registry_hook(DECODERS, unknown="keep")

d = json.loads(
    '{"x": {"__type__": "future_type", "value": 100}}',
    object_hook=lenient_hook,
)

d


{'x': {'__type__': 'future_type', 'value': 100}}

### Strict mode


In [60]:
strict_hook = make_registry_hook(DECODERS, unknown="error")

try:
    json.loads(
        '{"x": {"__type__": "future_type", "value": 100}}',
        object_hook=strict_hook,
    )
except ValueError as ex:
    print("Rejected:", ex)


Rejected: unknown tagged type: 'future_type'


---

# Problem 15 — Add Schema Versions

Custom JSON structures often evolve.

Suppose version 1 encoded a user like this:

```json
{
    "__type__": "user",
    "__version__": 1,
    "name": "Ada Lovelace",
    "id": 10
}
```

Version 2 now stores first and last names separately:

```json
{
    "__type__": "user",
    "__version__": 2,
    "first_name": "Ada",
    "last_name": "Lovelace",
    "user_id": 10
}
```

We want both forms to produce the same Python object.


In [61]:
@dataclass(frozen=True)
class User:
    first_name: str
    last_name: str
    user_id: int


### Step 1 — Decode version 1


In [62]:
def decode_user_v1(obj):
    first, separator, last = obj["name"].partition(" ")

    if not separator:
        last = ""

    return User(
        first_name=first,
        last_name=last,
        user_id=obj["id"],
    )


### Step 2 — Decode version 2


In [63]:
def decode_user_v2(obj):
    return User(
        first_name=obj["first_name"],
        last_name=obj["last_name"],
        user_id=obj["user_id"],
    )


### Step 3 — Dispatch by schema version


In [64]:
def versioned_user_decoder(obj):
    if obj.get("__type__") != "user":
        return obj

    version = obj.get("__version__")

    if version == 1:
        return decode_user_v1(obj)

    if version == 2:
        return decode_user_v2(obj)

    raise ValueError(f"unsupported user schema version: {version!r}")


In [65]:
v1 = '''
{
    "__type__": "user",
    "__version__": 1,
    "name": "Ada Lovelace",
    "id": 10
}
'''

v2 = '''
{
    "__type__": "user",
    "__version__": 2,
    "first_name": "Ada",
    "last_name": "Lovelace",
    "user_id": 10
}
'''

u1 = json.loads(v1, object_hook=versioned_user_decoder)
u2 = json.loads(v2, object_hook=versioned_user_decoder)

u1, u2


(User(first_name='Ada', last_name='Lovelace', user_id=10),
 User(first_name='Ada', last_name='Lovelace', user_id=10))

In [66]:
check(u1 == u2)
print("Problem 15 solved.")


Problem 15 solved.


### Takeaway

Version information belongs in the data when the interpretation of the data can change over time.


---

# Problem 16 — A Nested Dataclass Exercise: Decode a Project

Now let's combine several ideas.

We want these Python classes:


In [67]:
@dataclass(frozen=True)
class Milestone:
    name: str
    due: date

@dataclass(frozen=True)
class Project:
    project_id: UUID
    owner: User
    milestones: tuple


The JSON will contain:

- a tagged UUID,
- a tagged user,
- several tagged milestones,
- and an outer tagged project.

This is exactly where bottom-up decoding becomes powerful.


### Step 1 — Add a milestone decoder


In [68]:
def decode_milestone(obj):
    due = obj["due"]

    if not isinstance(due, date):
        raise TypeError("milestone.due must already be a date")

    return Milestone(
        name=obj["name"],
        due=due,
    )


### Step 2 — Add a project decoder


In [69]:
def decode_project(obj):
    project_id = obj["project_id"]
    owner = obj["owner"]
    milestones = obj["milestones"]

    if not isinstance(project_id, UUID):
        raise TypeError("project_id must already be UUID")

    if not isinstance(owner, User):
        raise TypeError("owner must already be User")

    if not all(isinstance(item, Milestone) for item in milestones):
        raise TypeError("every milestone must already be Milestone")

    return Project(
        project_id=project_id,
        owner=owner,
        milestones=tuple(milestones),
    )


### Step 3 — Create a dedicated registry

This time we use a strict registry because an unknown custom type should be considered an error.


In [70]:
PROJECT_DECODERS = {
    "uuid": decode_uuid_record,
    "date": decode_date_record,
    "user": lambda obj: (
        decode_user_v1(obj)
        if obj.get("__version__") == 1
        else decode_user_v2(obj)
        if obj.get("__version__") == 2
        else (_ for _ in ()).throw(
            ValueError(f"unsupported user version: {obj.get('__version__')!r}")
        )
    ),
    "milestone": decode_milestone,
    "project": decode_project,
}

project_hook = make_registry_hook(
    PROJECT_DECODERS,
    unknown="error",
)


In [71]:
project_json = '''
{
    "__type__": "project",
    "project_id": {
        "__type__": "uuid",
        "value": "aaaaaaaa-bbbb-cccc-dddd-eeeeeeeeeeee"
    },
    "owner": {
        "__type__": "user",
        "__version__": 2,
        "first_name": "Grace",
        "last_name": "Hopper",
        "user_id": 99
    },
    "milestones": [
        {
            "__type__": "milestone",
            "name": "Prototype",
            "due": {
                "__type__": "date",
                "value": "2026-09-01"
            }
        },
        {
            "__type__": "milestone",
            "name": "Launch",
            "due": {
                "__type__": "date",
                "value": "2026-10-15"
            }
        }
    ]
}
'''

project = json.loads(project_json, object_hook=project_hook)
project


Project(project_id=UUID('aaaaaaaa-bbbb-cccc-dddd-eeeeeeeeeeee'), owner=User(first_name='Grace', last_name='Hopper', user_id=99), milestones=(Milestone(name='Prototype', due=datetime.date(2026, 9, 1)), Milestone(name='Launch', due=datetime.date(2026, 10, 15))))

In [72]:
check(isinstance(project, Project))
check(isinstance(project.owner, User))
check(all(isinstance(m, Milestone) for m in project.milestones))
print("Problem 16 solved.")


Problem 16 solved.


---

# Problem 17 — Combine Duplicate-Key Detection with Custom Decoding

There is an important interaction between the two object hooks.

If both are supplied:

```python
object_hook=...
object_pairs_hook=...
```

then `object_pairs_hook` takes precedence.

So if we need:

- duplicate-key rejection, **and**
- custom tagged-object decoding,

we should combine both operations in one pairs hook.


### Step 1 — Build the object safely


In [73]:
def pairs_to_unique_dict(pairs):
    obj = {}

    for key, value in pairs:
        if key in obj:
            raise ValueError(f"duplicate key: {key!r}")
        obj[key] = value

    return obj


### Step 2 — After building the dictionary, run the project decoder


In [74]:
def strict_project_pairs_hook(pairs):
    obj = pairs_to_unique_dict(pairs)
    return project_hook(obj)


In [75]:
project_2 = json.loads(
    project_json,
    object_pairs_hook=strict_project_pairs_hook,
)

project_2


Project(project_id=UUID('aaaaaaaa-bbbb-cccc-dddd-eeeeeeeeeeee'), owner=User(first_name='Grace', last_name='Hopper', user_id=99), milestones=(Milestone(name='Prototype', due=datetime.date(2026, 9, 1)), Milestone(name='Launch', due=datetime.date(2026, 10, 15))))

In [76]:
check(project_2 == project)
print("Problem 17 solved.")


Problem 17 solved.


Now a duplicate key anywhere in the JSON tree will be rejected.


In [77]:
bad = '''
{
    "__type__": "date",
    "value": "2026-01-01",
    "value": "2027-01-01"
}
'''

try:
    json.loads(
        bad,
        object_pairs_hook=strict_project_pairs_hook,
    )
except ValueError as ex:
    print("Rejected:", ex)


Rejected: duplicate key: 'value'


---

# Problem 18 — Validate the Exact Shape of a Tagged Object

A tag alone may not be enough.

Suppose we expect a UUID object to contain exactly:

```text
__type__
value
```

If extra fields appear, we may want to reject them instead of silently ignoring them.

This is useful in strict protocols because unexpected fields can reveal schema drift.


In [78]:
def decode_uuid_strict(obj):
    expected = {"__type__", "value"}
    actual = set(obj)

    if actual != expected:
        missing = expected - actual
        extra = actual - expected

        raise ValueError(
            f"invalid UUID record; missing={sorted(missing)}, extra={sorted(extra)}"
        )

    return UUID(obj["value"])


In [79]:
good = '''
{
    "__type__": "uuid",
    "value": "12345678-1234-5678-1234-567812345678"
}
'''

json.loads(
    good,
    object_hook=lambda obj: (
        decode_uuid_strict(obj)
        if obj.get("__type__") == "uuid"
        else obj
    ),
)


UUID('12345678-1234-5678-1234-567812345678')

Now add an unexpected field.


In [80]:
bad = '''
{
    "__type__": "uuid",
    "value": "12345678-1234-5678-1234-567812345678",
    "comment": "unexpected"
}
'''

try:
    json.loads(
        bad,
        object_hook=lambda obj: (
            decode_uuid_strict(obj)
            if obj.get("__type__") == "uuid"
            else obj
        ),
    )
except ValueError as ex:
    print("Rejected:", ex)


Rejected: invalid UUID record; missing=[], extra=['comment']


---

# Problem 19 — Decode Tagged Decimal Values and Ordinary Decimal Numbers

There are two common approaches for decimal data.

### Approach A

Use ordinary JSON numeric tokens:

```json
{"price": 12.50}
```

and `parse_float=Decimal`.

### Approach B

Use an explicit tagged representation:

```json
{
    "__type__": "decimal",
    "value": "12.50"
}
```

Why might we prefer the tagged version?

Because it preserves the *semantic type* explicitly, even when the value happens to be an integer-like decimal such as `"10.00"`.


In [81]:
def decode_decimal_record(obj):
    if obj.get("__type__") == "decimal":
        return Decimal(obj["value"])
    return obj


In [82]:
j = '''
{
    "ordinary": 10.00,
    "explicit": {
        "__type__": "decimal",
        "value": "10.00"
    }
}
'''

d = json.loads(
    j,
    object_hook=decode_decimal_record,
    parse_float=Decimal,
)

d


{'ordinary': Decimal('10.00'), 'explicit': Decimal('10.00')}

In [83]:
check(d["ordinary"] == Decimal("10.00"))
check(d["explicit"] == Decimal("10.00"))
check(d["explicit"].as_tuple().exponent == -2)

print("Problem 19 solved.")


Problem 19 solved.


The tagged form is more verbose, but it can make a cross-language schema more explicit.


---

# Problem 20 — Create a Reusable `JSONDecoder` Class

So far, every `json.loads` call repeats configuration:

```python
parse_float=Decimal
parse_constant=...
object_pairs_hook=...
```

If the same policy is used throughout an application, a custom `JSONDecoder` subclass can package it.


We will build a decoder for the project protocol that:

- rejects duplicate keys,
- rejects `NaN` and infinities,
- reconstructs project-related tagged objects.


In [84]:
class ProjectJSONDecoder(json.JSONDecoder):
    def __init__(self, *args, **kwargs):
        kwargs.pop("object_hook", None)
        kwargs.pop("object_pairs_hook", None)
        kwargs.pop("parse_constant", None)

        super().__init__(
            *args,
            object_pairs_hook=strict_project_pairs_hook,
            parse_constant=reject_constant,
            **kwargs,
        )


In [85]:
decoded = json.loads(
    '''{
        "__type__": "date",
        "value": "2026-12-31"
    }''',
    cls=ProjectJSONDecoder,
)

decoded


datetime.date(2026, 12, 31)

The previous example used a `date`, which is in our project registry.

Now decode the complete project again.


In [86]:
decoded_project = json.loads(
    project_json,
    cls=ProjectJSONDecoder,
)

decoded_project


Project(project_id=UUID('aaaaaaaa-bbbb-cccc-dddd-eeeeeeeeeeee'), owner=User(first_name='Grace', last_name='Hopper', user_id=99), milestones=(Milestone(name='Prototype', due=datetime.date(2026, 9, 1)), Milestone(name='Launch', due=datetime.date(2026, 10, 15))))

In [87]:
check(decoded_project == project)
print("Problem 20 solved.")


Problem 20 solved.


---

# Problem 21 — Use `json.load` Instead of `json.loads`

The custom hooks work with both:

- `json.loads` — JSON text already in memory,
- `json.load` — JSON read from a file-like object.

We can demonstrate this without creating a second file by using `StringIO`.


In [88]:
from io import StringIO


In [89]:
stream = StringIO('''
{
    "__type__": "fraction",
    "numerator": 11,
    "denominator": 20
}
''')


In [90]:
fraction_hook = make_registry_hook(
    {"fraction": decode_fraction_record},
    unknown="error",
)

value = json.load(
    stream,
    object_hook=fraction_hook,
)

value


Fraction(11, 20)

In [91]:
check(value == Fraction(11, 20))
print("Problem 21 solved.")


Problem 21 solved.


---

# Problem 22 — Diagnose Malformed JSON Separately from Schema Errors

There are two different failure categories:

### Parsing error

The text is not valid JSON.

Example:

```json
{"a": 1,}
```

### Interpretation / schema error

The JSON is syntactically valid, but our custom protocol rejects it.

Example:

```json
{
    "__type__": "fraction",
    "numerator": 1,
    "denominator": 0
}
```

Good applications often distinguish these cases.


### Step 1 — Inspect a `JSONDecodeError`


In [92]:
try:
    json.loads('{"a": 1,}')
except json.JSONDecodeError as ex:
    print("message :", ex.msg)
    print("line    :", ex.lineno)
    print("column  :", ex.colno)
    print("position:", ex.pos)


message : Illegal trailing comma before end of object
line    : 1
column  : 8
position: 7


### Step 2 — Create a stricter fraction decoder


In [93]:
def strict_fraction_decoder(obj):
    if obj.get("__type__") != "fraction":
        return obj

    numerator = obj["numerator"]
    denominator = obj["denominator"]

    if type(numerator) is not int:
        raise TypeError("numerator must be an int")

    if type(denominator) is not int:
        raise TypeError("denominator must be an int")

    if denominator == 0:
        raise ValueError("denominator cannot be zero")

    return Fraction(numerator, denominator)


In [94]:
try:
    json.loads(
        '''
        {
            "__type__": "fraction",
            "numerator": 1,
            "denominator": 0
        }
        ''',
        object_hook=strict_fraction_decoder,
    )
except ValueError as ex:
    print("Schema error:", ex)


Schema error: denominator cannot be zero


A `JSONDecodeError` means the JSON parser could not construct the basic JSON value tree.

A later `ValueError` or `TypeError` from our hook means parsing succeeded, but our application rejected the interpreted structure.


---

# Problem 23 — Protect Against Oversized Input Before Parsing

Custom hooks do not automatically solve resource-exhaustion problems.

For example, `json.loads` has no universal maximum-document-size argument.

One simple application-level guard is to check the input size before parsing.


In [95]:
def loads_with_limit(text, *, max_bytes, **kwargs):
    size = len(text.encode("utf-8"))

    if size > max_bytes:
        raise ValueError(
            f"JSON input is too large: {size} bytes > {max_bytes} bytes"
        )

    return json.loads(text, **kwargs)


In [96]:
small = '{"message": "hello"}'

loads_with_limit(
    small,
    max_bytes=100,
)


{'message': 'hello'}

In [97]:
large = '{"message": "' + ("x" * 1000) + '"}'

try:
    loads_with_limit(
        large,
        max_bytes=100,
    )
except ValueError as ex:
    print("Rejected:", ex)


Rejected: JSON input is too large: 1015 bytes > 100 bytes


This is only one defensive layer.

Real systems may additionally use:

- HTTP/body-size limits,
- schema validation,
- maximum nesting policies,
- timeouts,
- memory limits,
- streaming parsers for very large documents.


---

# Problem 24 — Avoid Dynamic Class Import from Untrusted JSON

A tempting design is:

```json
{
    "__class__": "some.module.SomeClass",
    "args": [...]
}
```

and then dynamically import whatever class name appears in the JSON.

That is a dangerous pattern for untrusted input.

The data should never get to choose arbitrary executable Python objects.

Instead, use a fixed allowlist.


In [98]:
@dataclass(frozen=True)
class Point:
    x: Decimal
    y: Decimal

@dataclass(frozen=True)
class Size:
    width: Decimal
    height: Decimal


In [99]:
def decode_point_class(obj):
    return Point(obj["x"], obj["y"])

def decode_size_class(obj):
    return Size(obj["width"], obj["height"])

ALLOWED_CLASSES = {
    "Point": decode_point_class,
    "Size": decode_size_class,
}


In [100]:
def safe_class_hook(obj):
    class_name = obj.get("__class__")

    if class_name is None:
        return obj

    decoder = ALLOWED_CLASSES.get(class_name)

    if decoder is None:
        raise ValueError(f"class is not allowed: {class_name!r}")

    return decoder(obj)


In [101]:
j = '''
{
    "position": {
        "__class__": "Point",
        "x": 1.5,
        "y": 2.25
    },
    "size": {
        "__class__": "Size",
        "width": 100.0,
        "height": 50.0
    }
}
'''

d = json.loads(
    j,
    object_hook=safe_class_hook,
    parse_float=Decimal,
)

d


{'position': Point(x=Decimal('1.5'), y=Decimal('2.25')),
 'size': Size(width=Decimal('100.0'), height=Decimal('50.0'))}

Now test a class name that is not on the allowlist.


In [102]:
try:
    json.loads(
        '{"__class__": "os.system", "command": "echo nope"}',
        object_hook=safe_class_hook,
    )
except ValueError as ex:
    print("Rejected safely:", ex)


Rejected safely: class is not allowed: 'os.system'


---

# Problem 25 — Build a Complete Strict Decoder Pipeline

Let's combine the ideas from the notebook.

We want a decoder for configuration files with these rules:

1. duplicate object keys are rejected;
2. non-standard constants are rejected;
3. JSON floating-point numbers become `Decimal`;
4. known custom tagged objects are decoded;
5. unknown custom tags are rejected.

Supported custom types:

- date
- datetime
- fraction
- UUID
- path
- coordinate
- job status


### Step 1 — Build the strict tagged-object hook


In [103]:
STRICT_DECODERS = {
    "date": decode_date_record,
    "datetime": decode_datetime_record,
    "fraction": decode_fraction_record,
    "uuid": decode_uuid_record,
    "path": decode_path_record,
    "coordinate": decode_coordinate_record,
    "job_status": decode_status_record,
}

strict_tag_hook = make_registry_hook(
    STRICT_DECODERS,
    unknown="error",
)


### Step 2 — Combine duplicate checking and custom decoding


In [104]:
def strict_pairs_hook(pairs):
    obj = pairs_to_unique_dict(pairs)
    return strict_tag_hook(obj)


### Step 3 — Package everything in a loader function


In [105]:
def strict_loads(text):
    return json.loads(
        text,
        object_pairs_hook=strict_pairs_hook,
        parse_float=Decimal,
        parse_constant=reject_constant,
    )


### Step 4 — Decode a realistic configuration


In [106]:
config_json = '''
{
    "application": {
        "name": "analytics-service",
        "enabled": true,
        "retry_delay": 0.25
    },
    "workspace": {
        "__type__": "path",
        "value": "data/analytics"
    },
    "instance_id": {
        "__type__": "uuid",
        "value": "f47ac10b-58cc-4372-a567-0e02b2c3d479"
    },
    "deployment_date": {
        "__type__": "date",
        "value": "2026-08-07"
    },
    "health": {
        "__type__": "job_status",
        "value": "running"
    },
    "sample_ratio": {
        "__type__": "fraction",
        "numerator": 1,
        "denominator": 10
    },
    "office": {
        "__type__": "coordinate",
        "latitude": 42.6977,
        "longitude": 23.3219
    }
}
'''

config = strict_loads(config_json)
config


{'application': {'name': 'analytics-service',
  'enabled': True,
  'retry_delay': Decimal('0.25')},
 'workspace': WindowsPath('data/analytics'),
 'instance_id': UUID('f47ac10b-58cc-4372-a567-0e02b2c3d479'),
 'deployment_date': datetime.date(2026, 8, 7),
 'health': <JobStatus.RUNNING: 'running'>,
 'sample_ratio': Fraction(1, 10),
 'office': Coordinate(latitude=Decimal('42.6977'), longitude=Decimal('23.3219'))}

In [107]:
check(config["application"]["retry_delay"] == Decimal("0.25"))
check(isinstance(config["workspace"], Path))
check(isinstance(config["instance_id"], UUID))
check(isinstance(config["deployment_date"], date))
check(config["health"] is JobStatus.RUNNING)
check(config["sample_ratio"] == Fraction(1, 10))
check(isinstance(config["office"], Coordinate))

print("Problem 25 solved.")


Problem 25 solved.


---

# Problem 26 — Round-Trip Test a Custom Type

If we control both encoding and decoding, a powerful test is:

```text
Python object
    -> JSON
    -> Python object
```

The final object should equal the original one.

Let's do that for `Coordinate`.


### Step 1 — Write an encoder function


In [108]:
def custom_encoder(obj):
    if isinstance(obj, Coordinate):
        return {
            "__type__": "coordinate",
            "latitude": obj.latitude,
            "longitude": obj.longitude,
        }

    if isinstance(obj, Decimal):
        return {
            "__type__": "decimal",
            "value": str(obj),
        }

    raise TypeError(
        f"Object of type {type(obj).__name__} is not JSON serializable"
    )


The `Coordinate` contains `Decimal` values, so those values also need a JSON representation.

Now we need a decoder registry that understands both tags.


In [109]:
def decode_decimal_tag(obj):
    return Decimal(obj["value"])

ROUNDTRIP_DECODERS = {
    "decimal": decode_decimal_tag,
    "coordinate": decode_coordinate_record,
}

roundtrip_hook = make_registry_hook(
    ROUNDTRIP_DECODERS,
    unknown="error",
)


### Step 2 — Encode


In [110]:
original = Coordinate(
    latitude=Decimal("42.6977"),
    longitude=Decimal("23.3219"),
)

encoded = json.dumps(
    original,
    default=custom_encoder,
    indent=2,
)

print(encoded)


{
  "__type__": "coordinate",
  "latitude": {
    "__type__": "decimal",
    "value": "42.6977"
  },
  "longitude": {
    "__type__": "decimal",
    "value": "23.3219"
  }
}


### Step 3 — Decode


In [111]:
restored = json.loads(
    encoded,
    object_hook=roundtrip_hook,
)

restored


Coordinate(latitude=Decimal('42.6977'), longitude=Decimal('23.3219'))

### Step 4 — Verify


In [112]:
check(restored == original)
print("Round trip succeeded.")
print("Problem 26 solved.")


Round trip succeeded.
Problem 26 solved.


---

# Problem 27 — Advanced Challenge: Decode an API Response Envelope

Let's finish with a larger example.

We want to decode an API response like this:

```json
{
    "__type__": "api_response",
    "request_id": {... UUID ...},
    "generated_at": {... datetime ...},
    "status": {... job_status ...},
    "payload": {
        "location": {... coordinate ...},
        "confidence": 0.975
    }
}
```

The outer custom object should become an `ApiResponse` dataclass.

The nested ordinary `payload` object should remain a dictionary, but its custom children should still be decoded.


In [113]:
@dataclass(frozen=True)
class ApiResponse:
    request_id: UUID
    generated_at: datetime
    status: JobStatus
    payload: dict


### Step 1 — Write the outer decoder

Because object hooks run bottom-up, we expect the UUID, datetime, job status, and coordinate to be decoded before the `api_response` object is handled.


In [114]:
def decode_api_response(obj):
    request_id = obj["request_id"]
    generated_at = obj["generated_at"]
    status = obj["status"]
    payload = obj["payload"]

    if not isinstance(request_id, UUID):
        raise TypeError("request_id must be UUID")

    if not isinstance(generated_at, datetime):
        raise TypeError("generated_at must be datetime")

    if not isinstance(status, JobStatus):
        raise TypeError("status must be JobStatus")

    if not isinstance(payload, dict):
        raise TypeError("payload must remain an ordinary dictionary")

    return ApiResponse(
        request_id=request_id,
        generated_at=generated_at,
        status=status,
        payload=payload,
    )


### Step 2 — Extend the registry


In [115]:
API_DECODERS = {
    "uuid": decode_uuid_record,
    "datetime": decode_datetime_record,
    "job_status": decode_status_record,
    "coordinate": decode_coordinate_record,
    "api_response": decode_api_response,
}

api_tag_hook = make_registry_hook(
    API_DECODERS,
    unknown="error",
)


### Step 3 — Combine with duplicate checking


In [116]:
def api_pairs_hook(pairs):
    obj = pairs_to_unique_dict(pairs)
    return api_tag_hook(obj)


### Step 4 — Decode


In [117]:
api_json = '''
{
    "__type__": "api_response",
    "request_id": {
        "__type__": "uuid",
        "value": "00000000-0000-0000-0000-000000000123"
    },
    "generated_at": {
        "__type__": "datetime",
        "value": "2026-08-07T14:30:00Z"
    },
    "status": {
        "__type__": "job_status",
        "value": "complete"
    },
    "payload": {
        "location": {
            "__type__": "coordinate",
            "latitude": 42.6977,
            "longitude": 23.3219
        },
        "confidence": 0.975
    }
}
'''

response = json.loads(
    api_json,
    object_pairs_hook=api_pairs_hook,
    parse_float=Decimal,
    parse_constant=reject_constant,
)

response


ApiResponse(request_id=UUID('00000000-0000-0000-0000-000000000123'), generated_at=datetime.datetime(2026, 8, 7, 14, 30, tzinfo=datetime.timezone.utc), status=<JobStatus.COMPLETE: 'complete'>, payload={'location': Coordinate(latitude=Decimal('42.6977'), longitude=Decimal('23.3219')), 'confidence': Decimal('0.975')})

### Step 5 — Verify every layer


In [118]:
check(isinstance(response, ApiResponse))
check(isinstance(response.request_id, UUID))
check(isinstance(response.generated_at, datetime))
check(response.status is JobStatus.COMPLETE)
check(isinstance(response.payload, dict))
check(isinstance(response.payload["location"], Coordinate))
check(response.payload["confidence"] == Decimal("0.975"))

print("Problem 27 solved.")


Problem 27 solved.


---

# Problem 28 — Final Design Exercise: Strict vs Flexible Decoders

There is no single universally correct decoding policy.

For example, consider an unknown tagged object:

```json
{
    "__type__": "new_feature",
    "value": 123
}
```

Two applications may make different choices.

### Flexible consumer

Keep the unknown dictionary unchanged.

This can help with forward compatibility.

### Strict consumer

Raise an error immediately.

This can help detect protocol mismatches early.

The important design lesson is not that one policy is always superior.

The important lesson is that the policy should be **intentional, explicit, documented, and tested**.


In [119]:
flexible = make_registry_hook(
    {"date": decode_date_record},
    unknown="keep",
)

strict = make_registry_hook(
    {"date": decode_date_record},
    unknown="error",
)


In [120]:
text = '''
{
    "feature": {
        "__type__": "new_feature",
        "value": 123
    }
}
'''

json.loads(text, object_hook=flexible)


{'feature': {'__type__': 'new_feature', 'value': 123}}

In [121]:
try:
    json.loads(text, object_hook=strict)
except ValueError as ex:
    print("Strict decoder rejected it:", ex)


Strict decoder rejected it: unknown tagged type: 'new_feature'


That trade-off is part of API and data-model design, not merely a Python implementation detail.


---

# Summary — What You Should Be Comfortable With

After working through these problems, you should be able to reason about custom JSON decoding in layers.

## Layer 1 — Standard JSON parsing

You know that JSON naturally maps into only a small set of Python types:

- object -> `dict`
- array -> `list`
- string -> `str`
- integer -> `int`
- decimal/exponent number -> `float` by default
- true/false -> `True` / `False`
- null -> `None`

## Layer 2 — Numeric hooks

You can customize numeric construction with:

- `parse_int`
- `parse_float`
- `parse_constant`

## Layer 3 — Object hooks

You can customize JSON objects with:

- `object_hook`
- `object_pairs_hook`

You also know that custom object processing happens **bottom-up**.

## Layer 4 — Application schemas

You can interpret explicit tagged schemas as:

- dates,
- datetimes,
- UUIDs,
- fractions,
- paths,
- enums,
- dataclasses,
- and nested domain objects.

## Layer 5 — Robustness

You can add:

- field validation,
- duplicate-key rejection,
- strict unknown-tag policies,
- schema versions,
- input-size limits,
- and allowlists for safe object reconstruction.

## Layer 6 — Reuse

You can package a policy in:

- a registry,
- a loader function,
- or a `JSONDecoder` subclass.


# Best-Practice Checklist

When designing a custom JSON decoder, ask:

1. **How does the JSON explicitly identify a custom type?**
2. **Could the type marker collide with normal user data?**
3. **What fields are required?**
4. **Are extra fields allowed?**
5. **What value types and ranges are valid?**
6. **What happens when the custom type is unknown?**
7. **Does the schema need a version?**
8. **Should duplicate keys be rejected?**
9. **Should decimal numbers use `Decimal`?**
10. **Should `NaN` and infinities be rejected?**
11. **Can nested children be decoded bottom-up?**
12. **Could the JSON trigger arbitrary code, imports, or constructors?**
13. **Do we need document-size or resource limits?**
14. **Are encoder/decoder round-trip tests available?**
15. **Are malformed JSON errors distinguished from schema errors?**


# Extra Practice Ideas

If you want to extend this notebook further, try these without looking back at the solutions:

- Add a tagged `timedelta`.
- Add a `Money(currency, amount)` dataclass.
- Require all decoded datetimes to be timezone-aware.
- Add a versioned `Coordinate` schema.
- Reject path values containing `..`.
- Decode a list of API responses and aggregate their confidence values.
- Build a decoder for an immutable application configuration.
- Create custom exception classes such as `UnknownTypeError` and `SchemaValidationError`.
- Add a strict decoder that reports duplicate keys and unknown tags with domain-specific error messages.
- Write an encoder for the project objects and round-trip an entire `Project`.
